<a href="https://colab.research.google.com/github/ShamGaneshan2008/.ipynb-files-/blob/main/Trading_Indicator.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **TRADING INDICATOR USING XGBOOST**

In [34]:
!pip install yfinance ta xgboost plotly -q

In [35]:
import yfinance as yf
import pandas as pd
import numpy as np
import ta
import plotly.graph_objects as go

from xgboost import XGBClassifier
from sklearn.metrics import accuracy_score, classification_report

In [36]:
ticker = "AAPL"

data = yf.download(
    ticker,
    period="1y",
    interval="1d",
    auto_adjust=True
)

data.head()

[*********************100%***********************]  1 of 1 completed


Price,Close,High,Low,Open,Volume
Ticker,AAPL,AAPL,AAPL,AAPL,AAPL
Date,,,,,
2025-09-24,251.381393,254.798778,250.116063,254.280687,42303700
2025-09-25,255.924591,256.223505,250.783594,252.278073,55202100
2025-09-26,254.519806,256.651929,252.845981,253.164811,46076300
2025-09-29,253.493607,254.061516,252.078834,253.623133,40127700
2025-09-30,253.692871,254.978117,252.178461,253.922020,37704300


In [37]:
data = data.dropna()

data.columns = data.columns.get_level_values(0)

print(data.shape)
print(data.columns)
print(data.isnull().sum())

(252, 5)
Index(['Close', 'High', 'Low', 'Open', 'Volume'], dtype='object', name='Price')
Price
Close     0
High      0
Low       0
Open      0
Volume    0
dtype: int64


In [38]:
data["RSI"] = ta.momentum.RSIIndicator(
    close=data["Close"],
    window=14
).rsi()

data["MACD"] = ta.trend.MACD(
    close=data["Close"],
).macd()

data["SMA_20"] = ta.trend.SMAIndicator(
    close=data["Close"],
    window=20
).sma_indicator()

data["SMA_50"] = ta.trend.SMAIndicator(
    close=data["Close"],
    window=50
).sma_indicator()

data["EMA_20"] = ta.trend.EMAIndicator(close=data["Close"], window=20).ema_indicator()

data["ATR"] = ta.volatility.AverageTrueRange(
    high=data["High"],
    low=data["Low"],
    close=data["Close"],
    window=14
).average_true_range()

data["Volume_Change"] = data["Volume"].pct_change()

data = data.dropna()

data.tail()

Price,Close,High,Low,Open,Volume,RSI,MACD,SMA_20,SMA_50,EMA_20,ATR,Volume_Change
Date,,,,,,,,,,,,
2026-09-18,336.130005,338.489990,332.529999,337.910004,86588200,64.300001,5.544682,322.639999,320.025286,325.002435,7.304394,1.359339
2026-09-21,338.980011,339.640015,333.049988,335.279999,34999200,66.366129,5.955871,324.121500,320.503920,326.333632,7.253367,-0.595797
2026-09-22,339.750000,345.339996,338.750000,340.140015,40711800,66.923105,6.271577,325.592000,320.958188,327.611382,7.205984,0.163221
2026-09-23,337.019989,341.799988,335.500000,341.079987,31587800,62.943193,6.229676,326.948000,321.406815,328.507439,7.141270,-0.224112
2026-09-24,336.970001,338.910004,334.299988,336.320007,13977649,62.869467,6.121867,328.123999,321.601859,329.313398,6.960466,-0.557498


In [39]:
future_return = data["Close"].shift(-1) / data["Close"] - 1

threshold = 0.005

data["Target"] = np.where(
    future_return > threshold, 1,
    np.where(future_return < threshold, -1, 0)
)

data = data.dropna()

data[["Close", "Target"]].tail(10)

Price,Close,Target
Date,,
2026-09-11,332.269989,-1
2026-09-14,333.079987,-1
2026-09-15,331.339996,-1
2026-09-16,332.410004,1
2026-09-17,337.000000,-1
2026-09-18,336.130005,1
2026-09-21,338.980011,-1
2026-09-22,339.750000,-1
2026-09-23,337.019989,-1


In [40]:
features = [
    "Open",
    "High",
    "Low",
    "Close",
    "Volume",
    "RSI",
    "MACD",
    "SMA_20",
    "SMA_50",
    "EMA_20",
    "ATR",
    "Volume_Change"
]

X = data[features]
y = data["Target"]

print("Features: ", X.shape)
print("Target: ", X.shape)
print("\nTarget distribution: ")
print(y.value_counts())

Features:  (203, 12)
Target:  (203, 12)

Target distribution: 
Target
-1    129
 1     73
 0      1
Name: count, dtype: int64


In [41]:
split = int(len(X) * 0.8)

X_train = X.iloc[:split]
X_test = X.iloc[split:]

y_train = y.iloc[:split]
y_test = y.iloc[split:]

print("Training data: ", X_train.shape)
print("Testing data: ", X_test.shape)

print("\nTraining period: ")
print(X_train.index[0], "->", X_train.index[-1])

print("\nTesting period:")
print(X_test.index[0], "→", X_test.index[-1])

Training data:  (162, 12)
Testing data:  (41, 12)

Training period: 
2025-12-03 00:00:00 -> 2026-07-28 00:00:00

Testing period:
2026-07-29 00:00:00 → 2026-09-24 00:00:00


In [54]:
model = XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    random_state=42
)

y_train_encoded = y_train.map({
    -1: 0,
    1: 1
})

model.fit(X_train, y_train_encoded)

print("Model training completed.")

Model training completed.


In [55]:
predictions = model.predict(X_test)

predictions = predictions - 1

print("First 20 predictions:")
print(predictions[:20])


First 20 predictions:
[-1 -1 -1  0 -1  0  0  0  0  0  0  0  0  0  0 -1  0  0  0 -1]


In [56]:
predictions = model.predict(X_test)
predictions = predictions - 1

accuracy = accuracy_score(y_test, predictions)

print(f"Accuracy: {accuracy:.2%}")

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        predictions,
        labels=[-1, 0, 1],
        target_names=["DOWN", "HOLD", "UP"],
        zero_division=0
    )
)

Accuracy: 34.15%

Classification Report:
              precision    recall  f1-score   support

        DOWN       0.67      0.52      0.58        27
        HOLD       0.00      0.00      0.00         1
          UP       0.00      0.00      0.00        13

    accuracy                           0.34        41
   macro avg       0.22      0.17      0.19        41
weighted avg       0.44      0.34      0.38        41



In [57]:
results = X_test.copy()

results["Actual"] = y_test
results["Predictions"] = predictions
results["Price"] = data.loc[X_test.index, "Close"]

results.head()

buy_signals = results[results["Prediction"] == 1]
sell_signals = results[results["Prediction"] == -1]

print("BUY signals :", len(buy_signals))
print("SELL signals:", len(sell_signals))

In [62]:
results = X_test.copy()

results["Actual"] = y_test
results["Predictions"] = predictions
results["Price"] = data.loc[X_test.index, "Close"]

buy_signals = results[results["Predictions"] == 1]
sell_signals = results[results["Predictions"] == -1]

# Note: Due to the model being trained on only two classes (DOWN and UP, mapped to -1 and 1),
# and the original 'HOLD' class (0) being absent from the training set, the model's predictions
# will only be -1 or 0 (corresponding to original -1 or 1). Therefore, 'buy_signals' (for predictions == 1)
# will likely be empty. This is a logical consequence of the model training setup where the 'HOLD' class
# was not represented in the training data.

fig = go.Figure()

fig.add_trace(go.Candlestick(
    x=results.index,
    open=results["Open"],
    high=results["High"],
    low=results["Low"],
    close=results["Close"],
    name="Price"
))

fig.add_trace(go.Scatter(
    x=buy_signals.index,
    y=buy_signals["Close"],
    mode="markers",
    name="BUY",
    marker=dict(
        symbol="triangle-up",
        size=12
    )
))

fig.add_trace(go.Scatter(
    x=sell_signals.index,
    y=sell_signals["Close"],
    mode="markers",
    name="SELL",
    marker=dict(
        symbol="triangle-down",
        size=12
    )
))

fig.update_layout(
    title=f"{ticker} — ML Trading Indicator",
    xaxis_title="Date",
    yaxis_title="Price",
    xaxis_rangeslider_visible=False,
    template="plotly_white"
)

fig.show()